# Lunar Lander — Cleanup & Swap Framework (Jupyter Day)

**Goal today:** Clean modular `db_manager_v2` + Jupiter swap client skeleton + `sell_half_investment` example.

Run cells in order. All code is self-contained for notebook development.

In [ ]:
# Cell 1: Imports & Setup
import sys
sys.path.append('.')  # so we can import local files

from db_manager_v2 import (
    engine, get_session,
    InvestmentCreate, SecurityCreate,
    insert_investment, insert_security,
    get_token_key, get_single_wallet_pk, is_number_wallets_1,
    ensure_asset_exists, aquire_wallet_key,
    record_partial_sell
)
from global_values import WORLD_STABLE_COIN, solana_tokens
import numpy as np
from datetime import datetime
import time

print('✅ Clean v2 modules imported')

## 1. Quick Sanity Checks (using v2)

In [ ]:
# Cell 2: Check wallet count
with get_session(engine) as session:
    res = is_number_wallets_1(session)
    print(res)

In [ ]:
# Cell 3: Get token key for USDC
token_key_res = get_token_key(WORLD_STABLE_COIN)
print(token_key_res)

## 2. Swap Client Framework Skeleton (Jupiter)

In [ ]:
# Cell 4: Minimal JupiterSwapClient (stub — expand with real jupiter-python-sdk)
from typing import Dict, Any
from result import Result, Ok, Err

class JupiterSwapClient:
    def __init__(self, rpc_url: str = None):
        self.rpc_url = rpc_url or 'https://api.mainnet-beta.solana.com'
        # TODO: initialize jupiter client or use requests to quote-api.jup.ag

    def get_quote(self, input_mint: str, output_mint: str, amount_lamports: int, slippage_bps: int = 50) -> Result[Dict]:
        """Get best route quote from Jupiter."""
        # Placeholder — replace with real call
        print(f'[Jupiter] Getting quote {input_mint[:4]}... → {output_mint[:4]}... | {amount_lamports} lamports')
        fake_quote = {
            'inputMint': input_mint,
            'outputMint': output_mint,
            'inAmount': str(amount_lamports),
            'outAmount': str(int(amount_lamports * 0.98)),  # fake 2% slippage
            'priceImpactPct': 0.1,
            'slippageBps': slippage_bps
        }
        return Ok(fake_quote)

    def execute_swap(self, quote: Dict, keypair) -> Result[str]:
        """Execute the swap and return tx signature."""
        print('[Jupiter] Executing swap... (stub)')
        fake_sig = '5xFakeTxSignatureForNotebook' + str(int(time.time()))
        return Ok(fake_sig)


jupiter = JupiterSwapClient()
print('✅ JupiterSwapClient ready (stub mode)')

## 3. Example: Sell Half an Investment (Core Workflow)

In [ ]:
# Cell 5: sell_half_investment using v2 + Jupiter client
def sell_half_investment(
    investment_id: int,
    jupiter_client: JupiterSwapClient,
    target_mint: str,           # what we are swapping into
    keypair = None              # your Solana keypair
) -> Result[Dict]:
    """
    Production-ready pattern for 'sell half'.
    1. Load investment
    2. Get quote from Jupiter
    3. Execute swap
    4. Record partial close + remaining investment via record_partial_sell
    """
    with get_session(engine) as session:
        try:
            # 1. Load investment
            from sqlalchemy import select
            from db_manager_v2 import Investment
            inv = session.execute(select(Investment).where(Investment.id == investment_id)).scalar_one()

            if inv.isClosed:
                return Err('Investment already closed')

            half_lamports = inv.amount // 2
            if half_lamports <= 0:
                return Err('Investment too small to split')

            # 2. Get Jupiter quote
            # TODO: get actual token mint from Asset relationship
            from_mint = WORLD_STABLE_COIN  # placeholder
            quote_res = jupiter_client.get_quote(from_mint, target_mint, half_lamports)
            if quote_res.is_error:
                return quote_res

            # 3. Execute swap
            tx_res = jupiter_client.execute_swap(quote_res.value, keypair)
            if tx_res.is_error:
                return tx_res
            tx_sig = tx_res.value
            sell_price = float(quote_res.value.get('outAmount', 0)) / 1_000_000  # rough USDC

            # 4. Record in DB (closes half, creates remaining open investment)
            record_res = record_partial_sell(
                session=session,
                investment_id=investment_id,
                sold_lamports=half_lamports,
                sell_tx_id=tx_sig,
                sell_price_usdc=sell_price,
                sell_fee_usdc=0.01
            )
            if record_res.is_error:
                return record_res

            return Ok({
                'status': 'success',
                'sold_lamports': half_lamports,
                'tx_signature': tx_sig,
                'remaining_investment_created': True
            })

        except Exception as e:
            return Err(str(e))


print('✅ sell_half_investment function defined')

### Next Steps in Notebook
- Replace stub Jupiter client with real `jupiter-python-sdk`
- Add real token mint lookup from Asset
- Implement `create_tax_record` call inside `record_partial_sell`
- Test end-to-end with a real (or virtual) investment

In [ ]:
# Cell 6: Example usage (commented — run after you have an investment_id)
# result = sell_half_investment(investment_id=12, jupiter_client=jupiter, target_mint='So11111111111111111111111111111111111111112')
# print(result)